# ETL text — 12 quyển / 3 NXB / 2 399 trang · viết lại sạch (lượt 8, 2026-09-01, D-158)

**Bấm Run all rồi đợi.** Notebook này CHỈ chạy đường TEXT (`--text-only` + `--build-bm25`)
— ảnh đã xong 12/12 quyển từ trước (D-121/D-124/D-131) và được khôi phục nguyên vẹn
từ Drive, không đụng lại.

## Vì sao viết lại (đọc trước khi chạy, đừng bỏ qua)

Ba lượt Colab trước (5,6,7) đều báo "xong" (exit code 0) nhưng **đo trực tiếp trên
`database/` sau khi tải về mới lộ ra vẫn CHƯA XONG**:

- **D-154/D-155**: gate bắt nghi công thức khớp xuyên dòng (`CO_DAU_BANG` dùng `\s*`
  nhận cả `\n`) → 527 chunk `gate_hit_no_line_located`, 0 merge thành công. Đã sửa
  regex + bọc try/except quanh lời gọi MinerU.
- **D-157**: lượt chạy lại vẫn 0 merge — vì bước "khôi phục checkpoint" kéo về một
  bản ĐÃ-XONG-CẢ-12-QUYỂN từ **lượt TRƯỚC khi có bản vá D-155**, và version-gate coi
  các trang đó là "đã xong" nên không OCR lại — 63,9% trang mang bug cũ sống sót
  qua cả lượt chạy tiếp theo.
- **D-158 — root cause CUỐI CÙNG**: trong 866 trang thực sự chạy code mới, **100%**
  (3714/3714) lượt gọi MinerU đều thất bại. Xác nhận bằng thực nghiệm: gọi lẻ
  `get_formula_client().read(crop)` KHÔNG bật `HF_HUB_OFFLINE` → tải model từ Hub,
  đọc đúng `CO₂`/`O₂`, KHÔNG lỗi gì. Nguyên nhân thật: `download_models.py
  --profile text-etl` **chưa từng tải model MinerU**, và notebook **chưa từng trỏ
  `FORMULA_MINERU_MODEL` sang bản local** — dưới `HF_HUB_OFFLINE=1`, lần lazy-load
  đầu tiên LUÔN raise. Hai lỗi đã sửa trong `master`.

**Notebook này viết lại để một lượt "Run all" là đủ, không cần biết trước những
chuyện trên** — mọi bước có cổng tự kiểm, thất bại thì DỪNG rõ ràng (không lặng lẽ
đi tiếp trên dữ liệu thiếu), và ô cuối cùng (mục 11) **đo trực tiếp trên DB vừa
dựng** để xác nhận thật — không chỉ tin "exit code 0".

## Bốn điều đã đổi so với mọi bản trước

1. `download_models.py --profile text-etl` nay tải **cả `bge-m3` LẪN MinerU**
   (~4,3 GB), và **thoát mã khác 0 nếu bất kỳ model nào tải hỏng** — không còn
   nuốt lỗi rồi báo "thành công".
2. Env runtime trỏ **`FORMULA_MINERU_MODEL` sang bản local** giống mọi model khác.
3. **`TEXT_EXTRACTION_VERSION` bump `v3_formula_hybrid` → `v4_formula_hybrid_fix`**
   (đã sửa trong `src/config.py`) — trang cũ dù mang version cũ đúng hay sai đều
   bị coi là cần OCR lại.
4. **`scripts/reset_text_all_books.py --all`** hạ cờ `text_indexed` ngay sau khi
   khôi phục checkpoint, cho mọi trang CHƯA đạt đúng version mới — an toàn vì
   version mới CHƯA từng có trang nào đạt được trước bản vá này (không lặp lại
   lỗ hổng D-157: tin version một mình khi hai lượt code khác nhau lỡ dùng
   CHUNG version). Mặc định còn resume-safe: không hạ cờ lại trang ĐÃ được OCR
   đúng trong chính lượt đang chạy. Cờ ẢNH không bị đụng.

Chi tiết đầy đủ: `document/decision_log.html` D-154..D-158, `CLAUDE.md`.


## 1. Clone repo

In [ ]:
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git
%cd project-bio-rag


In [ ]:
!git log --oneline -3


## 2. Cài dependencies

`mineru_vl_utils` + ghim `transformers>=4.49,<5` NGAY SAU `requirements.txt`
(D-101: transformers 5.x nạp hỏng lm_head của model họ Qwen2-VL — MinerU dùng
kiến trúc này — sinh token rác thay vì đọc kém). `poppler-utils` chỉ còn cần cho
đường upload PDF legacy; cài luôn cho chắc, nhẹ.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --upgrade mineru_vl_utils "transformers>=4.49,<5"
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-vie

import transformers
print("transformers:", transformers.__version__)
_v = transformers.__version__.split(".")
assert _v[0] != "5" or int(_v[1]) == 0, "Ghim sai - kiem lai transformers>=4.49,<5 co hieu luc chua"


In [ ]:
!tesseract --version | head -1 && tesseract --list-langs | grep -x vie


## 3. Secret + env cơ bản (đặt TRƯỚC khi tải model)

`HF_TOKEN` lấy từ Colab Secrets. Mở tab 🔑 (Secrets) bên trái, thêm khoá
`HF_TOKEN`, bật *Notebook access*.

In [ ]:
import os, multiprocessing
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)
os.environ["USE_GPU"] = "true"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| CPU cores:", n)


## 4. Tải model về `./models` (chạy ONLINE, trước khi bật offline)

Chỉ cần đúng 2 model cho `--text-only` + hybrid công thức:

| model | dùng cho | kích thước |
|---|---|---|
| `BAAI/bge-m3` | embedding chunk text | ~2 GB |
| `opendatalab/MinerU2.5-Pro-2605-1.2B` | hybrid OCR công thức (D-56/D-144) | ~2,3 GB |

**D-158: trước đây bước này CHỈ tải `bge-m3`** — MinerU chưa từng nằm trong
`download_models.py`, nên dưới `HF_HUB_OFFLINE=1` (đặt ở mục 5), lần lazy-load
MinerU đầu tiên LUÔN thất bại (100% deterministic, đo được 3714/3714 lần ở lượt
trước). Đã sửa `download_models.py`. Ô dưới dùng `subprocess` (không phải `!`)
để đọc được MÃ THOÁT thật và DỪNG ngay nếu tải hỏng — đừng chạy ETL trên model
thiếu.

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-u", "./src/utils/download_models.py",
                    "--save_dir", "./models", "--profile", "text-etl"])
if r.returncode != 0:
    raise RuntimeError(
        f"Tai model that bai (ma thoat {r.returncode}) - DUNG o day, dung chay "
        "ETL khi model chua du. Xem log o tren de biet model nao hong.")
print("Tai model xong, ma thoat 0.")


In [ ]:
import os

for ten, thu_muc in [("bge-m3", "bge-m3"),
                     ("MinerU2.5-Pro-2605-1.2B", "MinerU2.5-Pro-2605-1.2B")]:
    p = f"./models/{thu_muc}"
    con = os.path.exists(f"{p}/config.json")
    kich_thuoc = sum(os.path.getsize(os.path.join(dp, f))
                     for dp, _, fs in os.walk(p) for f in fs) / 1e9 if os.path.isdir(p) else 0
    print(f"{ten:28s} config.json={con}  ~{kich_thuoc:.2f} GB")
    assert con, f"{ten}: khong thay config.json o {p} - tai model that bai that su"


## 5. Mount Drive + đường dẫn

Hai đường dẫn Drive **KHÔNG được nhầm nhau** (bài học D-157 — chính vì lẫn lộn
checkpoint CŨ và MỚI mà một lượt chạy tưởng-thành-công lại mang bug):

- `DRIVE_CHECKPOINT_ROOT` — checkpoint CŨ (lượt 5-7), **chỉ ĐỌC**, dùng để khôi
  phục ẢNH (đã xong, đừng dựng lại tốn 5-6 giờ).
- `DRIVE_SYNC_ROOT` — thư mục MỚI, đặt tên riêng cho lượt này
  (`database_checkpoints_v4_formula_fix`), **chỉ GHI** — mọi thứ lượt này tạo ra
  nằm ở đây, không bao giờ trộn với checkpoint cũ.

Ô dưới còn đặt `PYTHONUNBUFFERED=1` + rút ngắn nhịp log tiến độ (`PROGRESS_
LOG_EVERY_PAGES=2`, `PROGRESS_LOG_EVERY_SECONDS=8`, mặc định 10 trang/30s) —
để dòng `% | ETA` từ `ProgressLogger` (`src/utils/progress.py`) hiện LIÊN TỤC ở
mọi cell chạy `main.py`/script con qua `subprocess`, thay vì đợi cả phút mới
thấy dòng mới (D-160).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

os.environ["RAG_DATA_DIR"] = "/content/drive/MyDrive/project_bio_rag/datasource_png"

# Index nang (ChromaDB + BM25) lam viec o DIA CUC BO trong suot phien - tranh
# SQLite qua Drive-FUSE de lock/hong (D-152). Dong bo SANG Drive sau moi quyen.
os.environ["RAG_DATABASE_DIR"] = "/content/database"

os.environ["RAG_MANIFEST_DIR"] = "/content/project-bio-rag/database/manifests"
os.environ["RAG_FINGERPRINT_DIR"] = "/content/project-bio-rag/database/fingerprints"

os.environ["FORMULA_HYBRID_ENABLED"] = "true"
# Khong dat TEXT_EXTRACTION_VERSION/IMAGE_EXTRACTION_VERSION o day - mac dinh
# trong src/config.py da dung (v4_formula_hybrid_fix / v19_pill_kernels, D-158).
# Dat lai o day de tiem an nguy co lech voi CLAUDE.md nhu D-152 tung dinh mac.

base = "/content/project-bio-rag/models"
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["FORMULA_MINERU_MODEL"] = f"{base}/MinerU2.5-Pro-2605-1.2B"

os.environ["HF_HUB_OFFLINE"] = "1"  # model da tai o muc 4

os.environ["DRIVE_CHECKPOINT_ROOT"] = "/content/drive/MyDrive/project_bio_rag/database_checkpoints"
os.environ["DRIVE_SYNC_ROOT"] = "/content/drive/MyDrive/project_bio_rag/database_checkpoints_v4_formula_fix"

# Hien % tien do SONG khi ETL chay o subprocess con (D-160): Python mac dinh
# CHAN BLOCK-BUFFER stdout/stderr khi dau ra khong phai tty that (dung voi
# subprocess.run khong capture, van co the bi giu lai vai KB moi flush tren
# mot so nen). PYTHONUNBUFFERED=1 ep flush tung dong ngay lap tuc. Rut ngan
# nhip log tu mac dinh 10 trang/30s xuong 2 trang/8s de % cap nhat lien tuc
# ke ca khi mot trang co goi MinerU (vai giay/dong).
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PROGRESS_LOG_EVERY_PAGES"] = "2"
os.environ["PROGRESS_LOG_EVERY_SECONDS"] = "8"

for key in ("RAG_DATA_DIR", "RAG_DATABASE_DIR", "RAG_MANIFEST_DIR",
            "RAG_FINGERPRINT_DIR", "FORMULA_HYBRID_ENABLED",
            "EMBEDDING_MODEL", "FORMULA_MINERU_MODEL", "HF_HUB_OFFLINE",
            "PYTHONUNBUFFERED", "PROGRESS_LOG_EVERY_PAGES",
            "PROGRESS_LOG_EVERY_SECONDS",
            "DRIVE_CHECKPOINT_ROOT", "DRIVE_SYNC_ROOT"):
    print(f"{key} = {os.environ[key]}")

import sys
sys.path.insert(0, "/content/project-bio-rag")
from src.config import TEXT_EXTRACTION_VERSION, IMAGE_EXTRACTION_VERSION
print("TEXT_EXTRACTION_VERSION  =", TEXT_EXTRACTION_VERSION)
print("IMAGE_EXTRACTION_VERSION =", IMAGE_EXTRACTION_VERSION)
assert TEXT_EXTRACTION_VERSION == "v4_formula_hybrid_fix", (
    "Version khong khop ky vong D-158 - kiem tra src/config.py da pull dung ban master chua")


### Giữ phiên sống

Colab ngắt phiên nếu 90 phút không TƯƠNG TÁC VỚI TRANG. Bấm nút này cho chắc
trước khi bắt đầu vòng lặp ETL dài.

In [ ]:
%%javascript
function KeepClicking(){
  var btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); console.log("Da bam connect luc " + new Date()); }
}
setInterval(KeepClicking, 60000);


## 6. Khôi phục ẢNH từ checkpoint Drive (CHỈ đọc, không chạy lại ETL ảnh)

Ảnh đã xong 12/12 quyển (3 881 doc, `v19_pill_kernels`) từ trước lượt hybrid công
thức — bước này khôi phục lại đúng những gì đã có, không tính toán gì mới. Nếu
không tìm thấy checkpoint nào, DỪNG (raise) — đi tiếp trên DB trống nghĩa là mất
luôn ảnh, phải rebuild 5-6 giờ.

**D-161: sao chép qua Drive-FUSE có thể rớt kết nối giữa chừng** (`OSError:
[Errno 107] Transport endpoint is not connected` — đã gặp thật trên chính bước
này). `_copy_resilient()` dưới đi TỪNG FILE (không gọi `shutil.copytree`
nguyên khối) và thử lại tối đa 5 lần/file trước khi báo lỗi rõ ràng — một file
rớt kết nối không huỷ phần cây đã sao chép được, và than phiền thật (không
phải OOM/GPU) hiện ngay trong thông báo lỗi thay vì traceback khó đọc. Hàm này
dùng lại ở mục 10/13 khi đồng bộ NGƯỢC lên Drive.

In [ ]:
import re
import shutil
import time
from pathlib import Path

local_db = Path(os.environ["RAG_DATABASE_DIR"])
ckpt_root = Path(os.environ["DRIVE_CHECKPOINT_ROOT"])
local_db.mkdir(parents=True, exist_ok=True)


def _copy_resilient(src: Path, dst: Path, tries: int = 5, delay: float = 5.0):
    """Sao chep src (file/thu muc) vao dst, thu lai TUNG FILE khi Drive FUSE
    rot ket noi giua chung (D-161: ENOTCONN 'Transport endpoint is not
    connected' da gap that o buoc nay). Di tung file thay vi goi
    shutil.copytree nguyen khoi, de mot file loi khong huy phan cay da sao
    chep duoc, va retry chi lam lai dung file do (khong phai lam lai tu dau)."""
    if src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for child in src.iterdir():
            _copy_resilient(child, dst / child.name, tries, delay)
        return
    for attempt in range(1, tries + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as exc:
            if attempt == tries:
                raise RuntimeError(
                    f"Sao chep '{src}' -> '{dst}' that bai sau {tries} lan "
                    f"({exc}). Drive FUSE co the da rot ket noi that su - chay "
                    "lai drive.mount('/content/drive', force_remount=True) o "
                    "muc 5 roi chay lai tu day.") from exc
            print(f"  [thu lai {attempt}/{tries}] {src.name}: {exc}")
            time.sleep(delay)


# text-only + build-bm25 khong dung database/images (4,6 GB anh snapshot) va
# khong dung manifests/fingerprints (da co san qua git clone).
EXCLUDE_ON_RESTORE = {"images", "review", "logs", "manifests", "fingerprints",
                     "image_review_manifest.jsonl", "processed_files.txt",
                     "processed_images.txt", "qa_figure_coverage.json",
                     "ablation_cache.json", "mm_retrieval_cache.json"}


def _tim_checkpoint_moi_nhat():
    """Chi tin thu muc co DONE.marker (copy do dang thi KHONG co marker nay).

    Uu tien thu muc KHONG theo mau `after_NN_...` (vd `final_with_bm25`) truoc -
    do la ban da xong CA 12 quyen + BM25, day du hon bat ky `after_NN` nao. Neu
    khong co ban "final" nao thi lay `after_NN` co so lon nhat.
    """
    if not ckpt_root.exists():
        return None
    ung_vien = []
    for d in ckpt_root.iterdir():
        if not (d.is_dir() and (d / "DONE.marker").exists()):
            continue
        m = re.match(r"after_(\d+)_", d.name)
        do_uu_tien = int(m.group(1)) if m else 10_000  # "final_*" thang moi after_NN
        ung_vien.append((do_uu_tien, d))
    if not ung_vien:
        return None
    return max(ung_vien, key=lambda x: x[0])[1]


nguon = _tim_checkpoint_moi_nhat()
if nguon is None:
    raise RuntimeError(
        f"KHONG tim thay checkpoint nao co DONE.marker trong {ckpt_root}. "
        "Dung o day - di tiep se tao DB TRONG, mat 3 881 doc anh da dung "
        "(phai build lai anh ~5-6 gio). Kiem tra lai DRIVE_CHECKPOINT_ROOT.")
print(f"Khoi phuc anh + trang thai tu: {nguon}")

for item in nguon.iterdir():
    if item.name == "DONE.marker" or item.name in EXCLUDE_ON_RESTORE:
        continue
    dich = local_db / item.name
    if dich.exists():
        shutil.rmtree(dich) if dich.is_dir() else dich.unlink()
    _copy_resilient(item, dich)
print(f"Da khoi phuc {sum(1 for _ in local_db.iterdir())} muc vao {local_db}")


## 7. Reset checkpoint TEXT cho toàn bộ 12 quyển (D-158)

Checkpoint vừa khôi phục ở mục 6 **mang theo trạng thái TEXT của lượt trước**.
Script dưới hạ cờ `text_indexed`/`text_extraction_version` cho mọi trang CHƯA
đạt đúng `TEXT_EXTRACTION_VERSION` hiện tại (`v4_formula_hybrid_fix`) — an toàn
vì chuỗi này CHƯA từng có trang nào đạt được trước bản vá D-158, nên mọi trang
cũ (đúng hay sai) đều bị chọn. Mặc định này còn **resume-safe**: nếu phiên bị
ngắt giữa chừng và bạn chạy lại từ mục 6 (trỏ `DRIVE_CHECKPOINT_ROOT` sang
`DRIVE_SYNC_ROOT` để lấy đúng tiến độ của lượt này), những trang ĐÃ lên
`v4_formula_hybrid_fix` sẽ KHÔNG bị hạ cờ lại — không lãng phí công đã làm.
**Cờ ẢNH không bị đụng** (đã kiểm bằng test,
`tests/test_reset_text_all_books.py`).

In [ ]:
r = subprocess.run([sys.executable, "-u", "scripts/reset_text_all_books.py", "--all"])
if r.returncode != 0:
    raise RuntimeError(f"Reset text that bai (ma thoat {r.returncode}) - xem log o tren.")


## 8. Kiểm tra nguồn trước khi chạy bất cứ thứ gì

Kỳ vọng (đo 2026-08-23, D-65) — **12 quyển, 2 399 trang, 0 khoảng trống**. Ô dưới
in `thieu: []` cho **mọi** quyển thì mới chạy tiếp.

In [ ]:
import torch
from src.etl.page_source import discover_page_sources

print("CUDA:", torch.cuda.is_available())
total = 0
for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
    numbers = source.page_numbers()
    gaps = [n for n in range(numbers[0], numbers[-1] + 1) if n not in set(numbers)]
    total += len(numbers)
    print(f"{source.name}: {len(numbers)} trang, {numbers[0]}..{numbers[-1]}, thieu: {gaps}")
    if gaps:
        raise RuntimeError(f"{source.name} thieu trang {gaps} - kiem tra lai RAG_DATA_DIR truoc khi chay tiep.")
print("TONG:", total, "| shape trang mau:", source.load(numbers[0]).shape)
assert total == 2399, f"Tong trang = {total}, ky vong 2399 (D-65) - kiem tra lai nguon."


## 9. Kiểm tra manifest (đã có sẵn qua git, KHÔNG build lại)

12 file manifest đã commit trong repo (`database/manifests/`, đi theo `git clone`
ở mục 1) — G1 đã PASS cả 12/12 quyển từ trước (D-149/D-156). Chạy lại
`--build-manifests` tốn ~50-60 phút OCR toàn bộ, không cần thiết trừ khi vừa sửa
code đọc mục lục/banner. Ô dưới chỉ ĐỌC LẠI, không OCR.

In [ ]:
import glob, json

files = sorted(glob.glob("database/manifests/*.json"))
print(f"{len(files)}/12 file manifest")
assert len(files) == 12, "Thieu manifest - kiem tra git clone da lay dung repo/branch chua."
for f in files:
    m = json.load(open(f, encoding="utf-8"))
    print(f"{m['book_id']:12s} {m['n_pages']:4d} trang | offset {m['page_offset']}")


## 10. ETL — TEXT (mục chính, "Run all rồi đợi")

OCR theo **vùng layout** (không phải cả trang) + hybrid MinerU cho vùng nghi công
thức → chunk → index vào `biology_text`. Chạy từng quyển một, đồng bộ tiến độ
lên `DRIVE_SYNC_ROOT` ngay sau mỗi quyển — nếu phiên bị ngắt giữa chừng, đổi
`DRIVE_CHECKPOINT_ROOT` sang `DRIVE_SYNC_ROOT` ở mục 5 rồi chạy lại từ mục 6 để
tiếp tục ĐÚNG lượt này (ảnh + trang text đã xong sẽ được giữ nguyên nhờ
checkpoint theo NỘI DUNG trang, không phải theo tên quyển) — đừng bỏ qua mục 7
khi resume, nó vẫn an toàn (chỉ hạ cờ trang CHƯA từng đạt version mới).

**Một quyển lỗi thì DỪNG cả vòng lặp** — không âm thầm bỏ qua rồi báo "xong" cho
11/12 quyển.

In [ ]:
import datetime
import shutil
from pathlib import Path

BOOKS = ["SGK_KHTN_6_KNTT", "SGK_KHTN_7_KNTT", "SGK_KHTN_8_KNTT", "SGK_KHTN_9_KNTT",
        "SGK_KHTN_6_CTST", "SGK_KHTN_7_CTST", "SGK_KHTN_8_CTST", "SGK_KHTN_9_CTST",
        "SGK_KHTN_6_CD", "SGK_KHTN_7_CD", "SGK_KHTN_8_CD", "SGK_KHTN_9_CD"]

local_db = Path(os.environ["RAG_DATABASE_DIR"])
sync_root = Path(os.environ["DRIVE_SYNC_ROOT"])
sync_root.mkdir(parents=True, exist_ok=True)

EXCLUDE = {"images", "review", "logs", "manifests", "fingerprints",
          "image_review_manifest.jsonl", "processed_files.txt",
          "processed_images.txt", "qa_figure_coverage.json",
          "ablation_cache.json", "mm_retrieval_cache.json"}


def dong_bo_len_drive(nhan: str):
    # _copy_resilient() dinh nghia o muc 6 - thu lai tung file khi Drive FUSE
    # rot ket noi (D-161), khong huy ca thu muc dang dong bo vi mot file loi.
    dich = sync_root / nhan
    if dich.exists():
        shutil.rmtree(dich)
    dich.mkdir(parents=True)
    for item in local_db.iterdir():
        if item.name in EXCLUDE:
            continue
        _copy_resilient(item, dich / item.name)
    (dich / "DONE.marker").write_text(
        f"{datetime.datetime.now().isoformat()}\n", encoding="utf-8")
    print(f"Da dong bo checkpoint -> {dich}")


for idx, book in enumerate(BOOKS, start=1):
    print(f"\n=== [{idx}/12] ({100*(idx-1)/12:.0f}%) {book} ===")
    r = subprocess.run([sys.executable, "-u", "main.py", "--text-only", "--book", book])
    print(f"{book} exit code: {r.returncode}")
    if r.returncode != 0:
        raise RuntimeError(
            f"{book} THAT BAI (ma {r.returncode}) - DUNG, kiem log truoc khi "
            "chay lai (checkpoint Drive cua cac quyen truoc van con).")
    dong_bo_len_drive(f"after_{idx:02d}_{book}")
    print(f"--- {idx}/12 ({100*idx/12:.0f}%) da xong ---")


In [ ]:
from src.etl import ProcessingStatus
from src.config import TEXT_EXTRACTION_VERSION
from src.etl.page_source import discover_page_sources

status = ProcessingStatus()
con_thieu = 0
for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
    thieu = status.pages_needing_text(source, required_version=TEXT_EXTRACTION_VERSION)
    if thieu:
        con_thieu += len(thieu)
        print(f"{source.name}: con thieu {len(thieu)} trang text -> {thieu[:10]}{'...' if len(thieu) > 10 else ''}")
print(f"\nTONG con thieu: {con_thieu} trang" + (" - OK, het" if con_thieu == 0 else " - CON SOT, xem tren"))


## 11. XÁC NHẬN kết quả — đo trực tiếp trên DB, không chỉ tin exit code

Ba lượt trước đều báo "exit code 0" mà vẫn hỏng (D-154..D-157) — bài học là
**đừng bao giờ tin exit code một mình**. Ô dưới mở thẳng `database/` vừa dựng,
đếm số liệu THẬT, và in PASS/FAIL rõ ràng.

In [ ]:
import collections
import chromadb

client = chromadb.PersistentClient(path=os.environ["RAG_DATABASE_DIR"])

loi = []

# --- 1) Version coverage: DU 2399 trang len dung version moi ---
status_col = client.get_collection("processing_status")
n_status = status_col.count()
recs = [json.loads(d) for d in status_col.get(include=["documents"], limit=n_status)["documents"]]
n_text_ok = sum(1 for r in recs if r.get("text_extraction_version") == TEXT_EXTRACTION_VERSION)
n_image_ok = sum(1 for r in recs if r.get("image_extraction_version") == IMAGE_EXTRACTION_VERSION)
print(f"processing_status: {n_status} trang")
print(f"  text  dung version {TEXT_EXTRACTION_VERSION!r}: {n_text_ok}/{n_status}")
print(f"  image dung version {IMAGE_EXTRACTION_VERSION!r}: {n_image_ok}/{n_status}")
if n_text_ok != 2399:
    loi.append(f"Text version coverage = {n_text_ok}/2399, khong dat 100%")
if n_image_ok != 2399:
    loi.append(f"Image version coverage = {n_image_ok}/2399 - ANH BI MAT/HONG, kiem tra lai muc 6")

# --- 2) Anh khong bi mat trong qua trinh reset/restore ---
img_col = client.get_collection("biology_images")
n_img = img_col.count()
print(f"biology_images: {n_img} doc (ky vong ~3881, D-121/124/131)")
if n_img < 3800:
    loi.append(f"biology_images chi con {n_img} doc, ky vong ~3881 - ANH CO THE DA BI MAT")

# --- 3) Formula hybrid: co it nhat mot merge THANH CONG chua ---
text_col = client.get_collection("biology_text")
n_chunk = text_col.count()
metas = text_col.get(include=["metadatas"], limit=n_chunk)["metadatas"]
dem = collections.Counter()
for m in metas:
    s = m.get("formula_hybrid_status") or ""
    for tok in s.split(","):
        dem[tok if tok else "(rong)"] += 1
print(f"\nbiology_text: {n_chunk} chunk")
print("formula_hybrid_status breakdown:")
for k, v in sorted(dem.items(), key=lambda kv: -kv[1]):
    print(f"  {k:32s} {v:6d}")

n_applied = dem.get("applied", 0) + dem.get("unmatched_count", 0)
n_failed = sum(v for k, v in dem.items() if k.startswith("mineru_call_failed"))
n_gate_miss = dem.get("gate_hit_no_line_located", 0)
if n_applied == 0:
    loi.append("0 chunk co status 'applied'/'unmatched_count' - CHUA CO MOT LAN MERGE MINERU THANH CONG NAO")
if n_gate_miss > 50:
    loi.append(f"gate_hit_no_line_located = {n_gate_miss}, cao bat thuong (ky vong gan 0 sau D-155) - kiem tra lai")

print(f"\napplied+unmatched_count : {n_applied}")
print(f"mineru_call_failed*     : {n_failed}")
print(f"gate_hit_no_line_located: {n_gate_miss}")

print("\n" + "=" * 60)
if loi:
    print("KET QUA: THAT BAI - dung dong bo BM25/final tin day la du lieu tot")
    for x in loi:
        print(f"  - {x}")
else:
    print("KET QUA: DAT - the du lieu dung nhu ky vong D-158, an toan de dung bao cao")
print("=" * 60)


## 12. Chỉ mục THƯA (BM25)

Chỉ chạy sau khi mục 11 in "ĐẠT" — BM25 dựng từ CHÍNH `biology_text`, nên chunk
lỗi ở mục 11 sẽ đi thẳng vào chỉ mục thưa nếu bỏ qua cảnh báo.

In [ ]:
r = subprocess.run([sys.executable, "-u", "main.py", "--build-bm25"])
if r.returncode != 0:
    raise RuntimeError(f"build-bm25 that bai (ma {r.returncode})")


## 13. Đồng bộ checkpoint CUỐI CÙNG (kèm BM25)

In [ ]:
dong_bo_len_drive("final_with_bm25")


## Xong

Nếu mục 11 in **ĐẠT**: text ETL hoàn tất đúng như thiết kế D-56/D-144/D-158 —
`database_checkpoints_v4_formula_fix/final_with_bm25` trên Drive là bản dùng
được. Việc tiếp theo (ngoài notebook này, xem CLAUDE.md):

- `bai_so` cho 8 quyển CD/CTST (spine chưa liền mạch, không cần chạy lại ETL).
- Bộ câu hỏi sinh từ HÌNH có người đối chiếu ảnh (MT4 vế multi-modal).
- Đo lại đầu-cuối trên corpus mới bằng `document/colab_runtime_eval.ipynb`
  (D-182: `build_testset.py`/`retrieval_benchmark.py`/`run_eval.py` — thay
  `recall_at_k`/`ablation.py`/`evaluator.py` đã xoá), cập nhật báo cáo nếu số đổi.

Nếu mục 11 in **THẤT BẠI**: đừng chạy tiếp mục 12/13 — copy nguyên output của
mục 11 lại cho phiên làm việc để chẩn đoán, đừng đoán.